# Cross-Validation Basics with Medical Data[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ki-smile/trustcv/blob/main/notebooks/01_CV_Basics.ipynb)## Learning Objectives- Understand why cross-validation is critical for ML- Implement basic CV methods (hold-out, k-fold, stratified)- Recognize and avoid common pitfalls- Apply CV to a heart disease prediction task---

## 1. Setup and InstallationFirst, let's install trustcv and import necessary libraries.

In [ ]:
# Install trustcv (uncomment for Colab)# !pip install trustcv# Standard importsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFoldfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import accuracy_score, roc_auc_score, confusion_matriximport warningswarnings.filterwarnings('ignore')# Set styleplt.style.use('seaborn-v0_8-darkgrid')sns.set_palette(['#870052', '#FF876F', '#4F0433', '#EDF4F4'])

## 2. Load Medical DatasetWe'll use the Heart Disease dataset - a classic ML dataset.

In [ ]:
# Load heart disease datafrom sklearn.datasets import load_breast_cancer# For demonstration, we'll use breast cancer dataset# In practice, you'd load actual heart disease datadata = load_breast_cancer()X = pd.DataFrame(data.data, columns=data.feature_names)y = pd.Series(data.target, name='diagnosis')print(f"Dataset shape: {X.shape}")print(f"Class distribution:")print(y.value_counts(normalize=True))# Display first few rowsX.head()

## 3. The Problem: Why Simple Train-Test Split Isn't EnoughLet's first see what happens with a simple train-test split.

In [ ]:
# Multiple random splits show variance in resultsscores_simple = []for seed in range(10):    X_train, X_test, y_train, y_test = train_test_split(        X, y, test_size=0.2, random_state=seed    )        model = RandomForestClassifier(n_estimators=100, random_state=42)    model.fit(X_train, y_train)    score = accuracy_score(y_test, model.predict(X_test))    scores_simple.append(score)plt.figure(figsize=(10, 5))plt.subplot(1, 2, 1)plt.plot(scores_simple, 'o-', color='#870052')plt.xlabel('Random Seed')plt.ylabel('Accuracy')plt.title('Variance in Simple Train-Test Split')plt.axhline(np.mean(scores_simple), color='#FF876F', linestyle='--', label=f'Mean: {np.mean(scores_simple):.3f}')plt.legend()plt.subplot(1, 2, 2)plt.hist(scores_simple, bins=10, color='#870052', alpha=0.7, edgecolor='black')plt.xlabel('Accuracy')plt.ylabel('Frequency')plt.title(f'Distribution (std: {np.std(scores_simple):.3f})')plt.tight_layout()plt.show()print(f"⚠️ Problem: Accuracy varies from {min(scores_simple):.3f} to {max(scores_simple):.3f}")print(f"This {(max(scores_simple) - min(scores_simple))*100:.1f}% difference could be clinically significant!")

## 4. K-Fold Cross-Validation: The SolutionK-Fold CV uses all data for both training and testing, reducing variance.

In [ ]:
# Implement K-Fold Cross-Validationdef perform_kfold_cv(X, y, n_splits=5):    kfold = KFoldMedical(n_splits=n_splits, shuffle=True, random_state=42)    model = RandomForestClassifier(n_estimators=100, random_state=42)        fold_scores = []    fold_details = []        for fold, (train_idx, test_idx) in enumerate(kfold.split(X)):        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]                model.fit(X_train, y_train)        score = accuracy_score(y_test, model.predict(X_test))        fold_scores.append(score)                fold_details.append({            'fold': fold + 1,            'train_size': len(train_idx),            'test_size': len(test_idx),            'accuracy': score,            'class_dist_test': y_test.value_counts(normalize=True).to_dict()        })        return fold_scores, fold_details# Run K-Fold CVkfold_scores, kfold_details = perform_kfold_cv(X, y)# Visualize resultsfig, axes = plt.subplots(1, 3, figsize=(15, 5))# Fold scoresaxes[0].bar(range(1, 6), kfold_scores, color='#870052')axes[0].axhline(np.mean(kfold_scores), color='#FF876F', linestyle='--', linewidth=2)axes[0].set_xlabel('Fold')axes[0].set_ylabel('Accuracy')axes[0].set_title(f'K-Fold Scores (Mean: {np.mean(kfold_scores):.3f})')# Compare with simple splitaxes[1].boxplot([scores_simple, kfold_scores], labels=['Simple Split', 'K-Fold CV'])axes[1].set_ylabel('Accuracy')axes[1].set_title('Variance Comparison')# Fold sizesfold_nums = [d['fold'] for d in kfold_details]train_sizes = [d['train_size'] for d in kfold_details]test_sizes = [d['test_size'] for d in kfold_details]axes[2].bar(fold_nums, train_sizes, label='Train', color='#870052', alpha=0.7)axes[2].bar(fold_nums, test_sizes, bottom=train_sizes, label='Test', color='#FF876F', alpha=0.7)axes[2].set_xlabel('Fold')axes[2].set_ylabel('Number of Samples')axes[2].set_title('Data Split per Fold')axes[2].legend()plt.tight_layout()plt.show()print(f"✅ K-Fold CV: {np.mean(kfold_scores):.3f} ± {np.std(kfold_scores):.3f}")print(f"✅ Variance reduced by {(np.std(scores_simple) - np.std(kfold_scores))/np.std(scores_simple)*100:.1f}%")

## 5. Stratified K-Fold: Handling Class ImbalanceMedical datasets often have class imbalance. Stratified K-Fold preserves class distribution.

In [ ]:
# Create imbalanced dataset# Simulate rare disease (10% positive cases)np.random.seed(42)imbalanced_idx = np.random.choice(len(y), size=200, replace=False)X_imb = X.iloc[imbalanced_idx]y_imb = y.iloc[imbalanced_idx]# Make it more imbalancedpositive_idx = np.where(y_imb == 1)[0]keep_positive = np.random.choice(positive_idx, size=20, replace=False)negative_idx = np.where(y_imb == 0)[0]final_idx = np.concatenate([keep_positive, negative_idx])X_imb = X_imb.iloc[final_idx]y_imb = y_imb.iloc[final_idx]print(f"Imbalanced dataset class distribution:")print(y_imb.value_counts(normalize=True))# Compare regular K-Fold vs Stratified K-Folddef compare_cv_methods(X, y):    regular_kfold = KFoldMedical(n_splits=5, shuffle=True, random_state=42)    stratified_kfold = StratifiedKFoldMedical(n_splits=5, shuffle=True, random_state=42)        regular_class_dist = []    stratified_class_dist = []        for train_idx, test_idx in regular_kfold.split(X):        test_dist = y.iloc[test_idx].value_counts(normalize=True)        regular_class_dist.append(test_dist.get(1, 0))  # Positive class ratio        for train_idx, test_idx in stratified_kfold.split(X, y):        test_dist = y.iloc[test_idx].value_counts(normalize=True)        stratified_class_dist.append(test_dist.get(1, 0))        return regular_class_dist, stratified_class_distreg_dist, strat_dist = compare_cv_methods(X_imb, y_imb)# Visualizefig, axes = plt.subplots(1, 2, figsize=(12, 5))# Class distribution per foldfolds = range(1, 6)true_ratio = y_imb.value_counts(normalize=True)[1]axes[0].plot(folds, reg_dist, 'o-', label='Regular K-Fold', color='#4F0433', linewidth=2)axes[0].plot(folds, strat_dist, 's-', label='Stratified K-Fold', color='#870052', linewidth=2)axes[0].axhline(true_ratio, color='#FF876F', linestyle='--', label=f'True Ratio: {true_ratio:.2%}')axes[0].set_xlabel('Fold')axes[0].set_ylabel('Positive Class Ratio')axes[0].set_title('Class Distribution Across Folds')axes[0].legend()axes[0].grid(True, alpha=0.3)# Variance comparisonmethods = ['Regular\nK-Fold', 'Stratified\nK-Fold']variances = [np.std(reg_dist), np.std(strat_dist)]colors = ['#4F0433', '#870052']bars = axes[1].bar(methods, variances, color=colors)axes[1].set_ylabel('Standard Deviation')axes[1].set_title('Variance in Class Distribution')# Add value labels on barsfor bar, val in zip(bars, variances):    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,                f'{val:.4f}', ha='center', va='bottom')plt.tight_layout()plt.show()print(f"⚠️ Regular K-Fold: Class ratio varies from {min(reg_dist):.2%} to {max(reg_dist):.2%}")print(f"✅ Stratified K-Fold: Class ratio stable from {min(strat_dist):.2%} to {max(strat_dist):.2%}")

## 6. Medical Context: Patient-Level ConsiderationsIn medical data, we often have multiple records per patient. Let's simulate this scenario.

In [ ]:
# Simulate patient data with multiple recordsnp.random.seed(42)n_patients = 100records_per_patient = np.random.randint(1, 6, n_patients)  # 1-5 records per patientpatient_data = []for patient_id in range(n_patients):    n_records = records_per_patient[patient_id]    for _ in range(n_records):        patient_data.append({            'patient_id': f'P{patient_id:03d}',            'age': np.random.randint(30, 80),            'bp': np.random.normal(120, 20),            'cholesterol': np.random.normal(200, 40),            'heart_disease': np.random.choice([0, 1], p=[0.7, 0.3])        })df_patients = pd.DataFrame(patient_data)print(f"Total records: {len(df_patients)}")print(f"Unique patients: {df_patients['patient_id'].nunique()}")print(f"\nRecords per patient distribution:")print(df_patients['patient_id'].value_counts().value_counts().sort_index())

In [ ]:
# Demonstrate data leakage problemfrom sklearn.model_selection import GroupKFoldX_patient = df_patients[['age', 'bp', 'cholesterol']]y_patient = df_patients['heart_disease']groups = df_patients['patient_id']# Wrong way: Regular K-Fold (patient records can be in both train and test)regular_kf = KFoldMedical(n_splits=5, shuffle=True, random_state=42)# Right way: Group K-Fold (all patient records stay together)group_kf = GroupKFoldMedical(n_splits=5)# Check for leakagedef check_patient_leakage(X, groups, cv_method):    leakage_found = False        for fold, (train_idx, test_idx) in enumerate(cv_method.split(X, groups=groups)):        train_patients = set(groups.iloc[train_idx])        test_patients = set(groups.iloc[test_idx])                overlap = train_patients.intersection(test_patients)        if overlap:            leakage_found = True            print(f"❌ Fold {fold+1}: {len(overlap)} patients in both train and test!")        else:            print(f"✅ Fold {fold+1}: No patient overlap")        return leakage_foundprint("Regular K-Fold (WRONG for grouped data):")regular_leakage = check_patient_leakage(X_patient, groups, regular_kf)print("\nGroup K-Fold (CORRECT for grouped data):")group_leakage = check_patient_leakage(X_patient, groups, group_kf)if regular_leakage:    print("\n⚠️ WARNING: Regular K-Fold causes data leakage with patient data!")    print("This leads to overly optimistic performance estimates.")

## 7. Using trustcv for Best PracticesNow let's use the trustcv library to handle all these considerations automatically.

In [ ]:
# Simulate using trustcv (actual import would be: from trustcv import MedicalValidator)class MedicalValidatorDemo:    """Simplified demo version of MedicalValidator"""        def __init__(self, method='stratified_kfold', n_splits=5, check_leakage=True):        self.method = method        self.n_splits = n_splits        self.check_leakage = check_leakage        def fit_validate(self, model, X, y, patient_ids=None):        # Determine best CV method        if patient_ids is not None and len(patient_ids.unique()) < len(patient_ids):            cv = GroupKFoldMedical(n_splits=self.n_splits)            cv_type = "Patient-Grouped K-Fold"            groups = patient_ids        elif len(np.unique(y)) == 2 and min(np.bincount(y)) / max(np.bincount(y)) < 0.3:            cv = StratifiedKFoldMedical(n_splits=self.n_splits, shuffle=True, random_state=42)            cv_type = "Stratified K-Fold (imbalanced classes detected)"            groups = None        else:            cv = KFoldMedical(n_splits=self.n_splits, shuffle=True, random_state=42)            cv_type = "Standard K-Fold"            groups = None                # Perform validation        scores = cross_val_score(model, X, y, cv=cv, groups=groups)                # Generate report        report = {            'method_used': cv_type,            'mean_accuracy': np.mean(scores),            'std_accuracy': np.std(scores),            '95_ci': (np.mean(scores) - 1.96*np.std(scores),                      np.mean(scores) + 1.96*np.std(scores)),            'fold_scores': scores,            'recommendations': self._generate_recommendations(scores, X, y)        }                return report        def _generate_recommendations(self, scores, X, y):        recommendations = []                if np.std(scores) > 0.05:            recommendations.append("High variance detected. Consider increasing sample size.")                if X.shape[0] < 10 * X.shape[1]:            recommendations.append(f"Low sample-to-feature ratio ({X.shape[0]}/{X.shape[1]}). Consider feature selection.")                if len(recommendations) == 0:            recommendations.append("Validation setup looks good!")                return recommendations# Use trustcvvalidator = MedicalValidatorDemo(check_leakage=True)model = RandomForestClassifier(n_estimators=100, random_state=42)# Validate with patient dataresult = validator.fit_validate(model, X_patient, y_patient, patient_ids=groups)print("=== Trustworthy Cross-Validation Report ===")print(f"\nMethod: {result['method_used']}")print(f"Performance: {result['mean_accuracy']:.3f} ± {result['std_accuracy']:.3f}")print(f"95% CI: [{result['95_ci'][0]:.3f}, {result['95_ci'][1]:.3f}]")print(f"\nFold Scores: {[f'{s:.3f}' for s in result['fold_scores']]}")print(f"\nRecommendations:")for rec in result['recommendations']:    print(f"  • {rec}")

## 8. Practical Exercise: Complete PipelineLet's put it all together with a complete ML pipeline.

In [ ]:
# Complete pipeline with best practicesfrom sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import Pipelinefrom sklearn.metrics import classification_report# Create pipelinepipeline = Pipeline([    ('scaler', StandardScaler()),    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))])# Use stratified K-fold for final evaluationskf = StratifiedKFoldMedical(n_splits=5, shuffle=True, random_state=42)# Store resultscv_results = {    'accuracy': [],    'sensitivity': [],    'specificity': [],    'auc': []}# Perform cross-validation with detailed metricsfor fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]        # Train    pipeline.fit(X_train, y_train)        # Predict    y_pred = pipeline.predict(X_test)    y_proba = pipeline.predict_proba(X_test)[:, 1]        # Calculate metrics    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()        cv_results['accuracy'].append(accuracy_score(y_test, y_pred))    cv_results['sensitivity'].append(tp / (tp + fn) if (tp + fn) > 0 else 0)    cv_results['specificity'].append(tn / (tn + fp) if (tn + fp) > 0 else 0)    cv_results['auc'].append(roc_auc_score(y_test, y_proba))# Create results DataFrameresults_df = pd.DataFrame(cv_results)results_df['fold'] = range(1, 6)# Visualize comprehensive resultsfig, axes = plt.subplots(2, 2, figsize=(12, 10))# Metrics across foldsmetrics = ['accuracy', 'sensitivity', 'specificity', 'auc']colors_list = ['#870052', '#FF876F', '#4F0433', '#EDF4F4']for idx, (metric, color) in enumerate(zip(metrics, colors_list)):    ax = axes[idx // 2, idx % 2]        ax.bar(results_df['fold'], results_df[metric], color=color, alpha=0.7)    ax.axhline(results_df[metric].mean(), color='red', linestyle='--',               label=f'Mean: {results_df[metric].mean():.3f}')    ax.set_xlabel('Fold')    ax.set_ylabel(metric.capitalize())    ax.set_title(f'{metric.capitalize()} Across Folds')    ax.set_ylim([0, 1])    ax.legend()    ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()# Final summaryprint("=== Final Cross-Validation Results ===")print("\nMetric          Mean ± Std        [95% CI]")print("-" * 50)for metric in metrics:    mean = results_df[metric].mean()    std = results_df[metric].std()    ci_lower = mean - 1.96 * std    ci_upper = mean + 1.96 * std    print(f"{metric.capitalize():12}    {mean:.3f} ± {std:.3f}      [{ci_lower:.3f}, {ci_upper:.3f}]")# Clinical interpretationprint("\n=== Clinical Interpretation ===")sens = results_df['sensitivity'].mean()spec = results_df['specificity'].mean()if sens > 0.9:    print(f"✅ High sensitivity ({sens:.1%}): Good for screening (few false negatives)")elif sens > 0.8:    print(f"⚠️ Moderate sensitivity ({sens:.1%}): May miss some positive cases")else:    print(f"❌ Low sensitivity ({sens:.1%}): Many false negatives")if spec > 0.9:    print(f"✅ High specificity ({spec:.1%}): Good for confirmation (few false positives)")elif spec > 0.8:    print(f"⚠️ Moderate specificity ({spec:.1%}): Some false alarms")else:    print(f"❌ Low specificity ({spec:.1%}): Many false positives")

## 9. Key Takeaways### ✅ Do's:1. **Always use cross-validation** for reliable performance estimates2. **Use Stratified K-Fold** for imbalanced medical datasets3. **Use Group K-Fold** when you have multiple records per patient4. **Report confidence intervals** along with mean performance5. **Check for data leakage** especially with patient data### ❌ Don'ts:1. **Don't rely on single train-test split** - high variance2. **Don't mix patient data** between train and test sets3. **Don't ignore class imbalance** - use stratification4. **Don't use LOOCV** with large datasets - computationally expensive5. **Don't forget clinical context** - statistical significance ≠ clinical significance### 🎯 Next Steps:- Try the Patient-Level CV notebook for grouped data- Explore Temporal CV for time-series medical data- Learn about Nested CV for hyperparameter tuning

## 10. Practice ExercisesTry these exercises to reinforce your learning:

In [ ]:
# Exercise 1: Compare different number of folds# TODO: Implement cross-validation with k = 3, 5, 10# What happens to variance as k increases?# Your code here# Exercise 2: Implement Leave-One-Out CV# TODO: Use LeaveOneOut from sklearn# Compare computation time with 5-fold CV# Your code here# Exercise 3: Handle severe class imbalance# TODO: Create dataset with 95% negative, 5% positive# Compare regular vs stratified CV# Your code here

---### 🎉 Congratulations!You've completed the Cross-Validation Basics tutorial. You now understand:- Why CV is critical for ML- How to implement different CV strategies- When to use each method- How to avoid common pitfalls**Next**: Continue with [02_Patient_Level_CV.ipynb](02_Patient_Level_CV.ipynb) to learn about handling grouped medical data.---*Created with trustcv - Trustworthy Cross-Validation Toolkit*